# ML-08 — Capstone Modeling Lane: A Learned Decline-Risk Queue vs the Rule

**Lane 2 — Refresh / Content Opportunity Scoring · Build phase.** The Week-4 rule (`visible × slipped × depth`) ranked a refresh queue at ~0.56 precision@50 on the full slice. This notebook trains the first learned model on the **same March-2026 page-level slice, the same `is_declining_label` proxy, and the same precision@K metric** — but on a **client-grouped holdout** — then puts model and baseline in one table and reads the errors before believing the score.

**The week's research paper is read** (`docs/flyrank-seo-research-march-2026.pdf`). Two things I carry into next week's methodology inspection: the paper's ML appendix trains on a *self-selected* active-content subset (sessions > 0, impressions > 0) with **80/20 random row splits** and reports **no p-values or intervals**, and its headline health-score importance is explicitly "descriptive" because the target is partly constructed from the inputs. This notebook deliberately avoids the same weaknesses where the warehouse lets me: a **grouped (client-level) split**, a **label that is never a feature**, and a **same-split baseline comparison**.

## 1. Method choice and why

My question is **yes/no with an observed label** — "will this page's impressions decline next half?" — and the output is a **score**, because the strategist acts on the *top of an ordered queue*, not on a flat flag (w02 framing). From this week's toolkit:

| Method | Verdict | Why (for this lane) |
|---|---|---|
| Logistic Regression | **Yes — start here** | Readable coefficients I can sanity-check against the w04 signal audit (position should matter, raw volume should not) |
| Decision Tree (depth 3) | **Yes — for readability** | A printable tree in the same spirit as the paper's appendix shallow tree; shows the interactions without a black box |
| Random Forest | **Yes — the candidate to beat** | Position × volume × engagement interactions are the messy part of this decision; 200 trees, depth 10 — modest, not gratuitous |
| Gradient Boosting | **No this week** | ~5 real signals and ~26k train rows; it would buy a couple of points on noise and hurt the "explain the model" goal. Revisit only if RF wins by a real margin and I still can't explain the loss |
| K-Means clustering | **No — different question** | Grouping pages does not tell a strategist what to do first; clustering is a w05-menu item but not this lane's deliverable |
| Permutation importance | **Yes — as a check** | After RF, shuffle each feature on the test split to see what the model actually leans on — and to catch anything suspiciously perfect (= leak) |

**Simplicity is a feature.** The bar is: *does a learned model beat the transparent rule on the same split and metric, and can I explain its mistakes?* Not: *does it add complexity?*

### The slice — identical to the w03/w04 data contract

One row = one page (`content_hash_id` under `client_hash_id`) at the 2026-03-31 decision moment. Features are known by that moment; the label compares second-half March impressions to first-half. No `trend_*` columns, no product flags, no future windows.

In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv("../../.env")
HF_TOKEN = os.environ.get("HF_token") or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF token found in .env"

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
SNAP = "DATE '2026-03-31'"

OUT_DIR = Path("../../").resolve() / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [2]:
page = con.sql(
    f"""
    WITH daily AS (
        SELECT *
        FROM {FACT}
        WHERE ga4_data_available IS TRUE
          AND gsc_data_available IS TRUE
    ),
    page AS (
        SELECT
            d.client_hash_id,
            d.content_hash_id,
            SUM(d.gsc_impressions) AS gsc_impressions_mar,
            AVG(CASE WHEN d.gsc_avg_position > 0 THEN d.gsc_avg_position END) AS avg_position_mar,
            SUM(d.ga4_sessions) AS sessions_mar,
            SUM(d.ga4_engaged_sessions) AS engaged_sessions_mar,
            SUM(CASE WHEN d.report_date <= DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN d.report_date >  DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_second_half
        FROM daily d
        GROUP BY 1, 2
        HAVING SUM(d.gsc_impressions) >= 100
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        DATE_DIFF('day', dc.content_created_date, {SNAP}) AS content_age_days,
        GREATEST(DATE_DIFF('day', dc.content_updated_date, {SNAP}), 0) AS days_since_last_update,
        p.gsc_impressions_mar,
        p.avg_position_mar,
        CASE WHEN p.sessions_mar > 0
             THEN 100.0 * p.engaged_sessions_mar / p.sessions_mar END AS engagement_rate_mar,
        CASE WHEN p.imp_second_half < p.imp_first_half THEN 1 ELSE 0 END AS is_declining_label
    FROM page p
    JOIN {DIM} dc USING (content_hash_id)
    """
).df()

# Canonical row order so tie-broken precision@K and the grouped split do not depend on scan order.
features = (page.dropna(subset=["avg_position_mar"])
            .sort_values(["client_hash_id", "content_hash_id"])
            .reset_index(drop=True))
print(f"Page-level slice: {len(features):,} rows, {features['client_hash_id'].nunique()} clients")
print(f"Observed declining base rate: {features['is_declining_label'].mean():.3f}")
print("Missing values:", int(features.isna().sum().sum()))
print("Duplicate content ids:", int(features['content_hash_id'].duplicated().sum()))

Page-level slice: 32,596 rows, 30 clients
Observed declining base rate: 0.268
Missing values: 26
Duplicate content ids: 0


## 2. Split design

**Grouped by client — a 20% client holdout (seed 42).**

- **Why grouped, not row-random:** the queue is consumed *per client* — a strategist acts inside one client's inventory. A row-random split puts pages of the same client in both train and test, so the model gets to peek at that client's pattern while being scored on it; that overstates how it would behave on a client it has never seen. Client-level grouping is the honest simulation of deployment (mirrors `make_client_aware_split` in `scripts/03_train_model.py`).
- **Why no time split:** the slice is one month (March 2026) — there is no earlier window to train on. The label proxy is already future-guarded by construction (second-half vs first-half impressions inside March, and `imp_trend_pct_mar` never enters the feature set). One honest note on window alignment: the March feature aggregates span the whole month, so they partly overlap the second half that defines the label — the w03 contract accepted this (only the label-derived `imp_trend_pct_mar` is excluded). Worth one sentence because an ML-engineering reviewer will likely probe it next week.
- **Baseline on the same split:** the rule has no fitted parameters, so it needs no training — but to compare fairly it is re-scored **on the same test clients only**. The w04 numbers (precision@50 ≈ 0.56, full slice) are context, not this comparison.
- **Reproducibility:** one fixed seed (42) everywhere; library version noted in the run output. Tree-ensemble numbers can shift a point or two between sklearn releases — that is normal and worth one sentence.

**One honest caveat.** With 30 clients in the slice, a single 6-client holdout carries real fold-to-fold noise. So section 3 pairs the seed-42 holdout table with a **repeated grouped-split check** (5 seeds, same metric, same split each time) and reports both. The single-holdout numbers are the headline; the repeated check is the sanity guard.

In [3]:
def client_grouped_split(df, frac=0.2, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    unique_clients = np.sort(df["client_hash_id"].drop_duplicates().to_numpy())  # canonical order -> split independent of scan order
    order = rng.permutation(len(unique_clients))
    n_test = max(1, int(round(len(unique_clients) * frac)))
    test_clients = set(unique_clients[order[:n_test]])
    is_test = df["client_hash_id"].isin(test_clients).to_numpy()
    is_train = ~is_test
    for label, mask in (("train", is_train), ("test", is_test)):
        assert df.loc[mask, "is_declining_label"].nunique() == 2, f"{label} has a single class"
    return is_train, is_test, test_clients


is_train, is_test, test_clients = client_grouped_split(features)
tr = features.loc[is_train].reset_index(drop=True)
te = features.loc[is_test].reset_index(drop=True)

print(f"Train: {len(tr):,} rows / {tr['client_hash_id'].nunique()} clients | base rate {tr['is_declining_label'].mean():.3f}")
print(f"Test:  {len(te):,} rows / {te['client_hash_id'].nunique()} clients | base rate {te['is_declining_label'].mean():.3f}")

Train: 28,101 rows / 24 clients | base rate 0.257
Test:  4,495 rows / 6 clients | base rate 0.342


In [4]:
CONTRACT_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "gsc_impressions_mar",
    "avg_position_mar",
    "engagement_rate_mar",
]


def build_matrix(df, features=CONTRACT_FEATURES):
    X = df[features].copy()
    X = X.astype(float)
    X["log_gsc_impressions_mar"] = np.log1p(X["gsc_impressions_mar"])  # heavy-tail compression of the contract field
    X["has_engagement"] = X["engagement_rate_mar"].notna().astype(int)  # NaN = sessions == 0, not "zero engagement"
    X["engagement_rate_mar"] = X["engagement_rate_mar"].fillna(0.0)
    return X[["content_age_days", "days_since_last_update", "avg_position_mar",
              "log_gsc_impressions_mar", "engagement_rate_mar", "has_engagement"]]


X_tr = build_matrix(tr)
X_te = build_matrix(te)
y_tr = tr["is_declining_label"].astype(int).to_numpy()
y_te = te["is_declining_label"].astype(int).to_numpy()

print("Feature matrix —", X_tr.shape, "train,", X_te.shape, "test")
print("Columns:", list(X_tr.columns))
print("No label-derived column present:", not any("trend" in c or "half" in c for c in X_tr.columns))

Feature matrix — (28101, 6) train, (4495, 6) test
Columns: ['content_age_days', 'days_since_last_update', 'avg_position_mar', 'log_gsc_impressions_mar', 'engagement_rate_mar', 'has_engagement']
No label-derived column present: True


## 3. Train + compare vs my baseline

Same data (March-2026 page slice), same label proxy, same metric (precision@K, with ROC-AUC as a secondary ranking check), **same client-held-out test**. The baseline row is the w04 rule re-scored on test only; the base-rate row is what random ranking would give on test. LR and RF hyper-parameters match the repo's reference pipeline (`scripts/03_train_model.py`); the tree is deliberately shallower (depth 3 vs the repo's 5) for readability (section 1) — so the numbers stay comparable to repo norms, with that one deliberate deviation. Note: the baseline row's ROC-AUC reflects its many tied scores; precision@K is the primary metric. After the single holdout below, a 5-seed repeated grouped-split cell checks whether the winner generalizes or just won its one draw.

In [5]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

KS = [10, 25, 50, 100]


def precision_at_k(y, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(y[order][:k].mean())


def encode_rule(df):  # the w04 rule, verbatim
    df = df.copy()
    vis = (df["gsc_impressions_mar"] >= 500).astype(int)
    depth = df["avg_position_mar"].clip(upper=50) / 50.0
    slip = (df["avg_position_mar"] >= 10).astype(int)
    df["score"] = vis * slip * depth
    df["queue_score"] = df["score"] + (df["gsc_impressions_mar"] / df["gsc_impressions_mar"].quantile(0.995)) * 1e-6
    return df


def evaluate(y, score):
    return {f"prec@{k}": precision_at_k(y, score, k) for k in KS} | {
        "roc_auc": roc_auc_score(y, score),
        "avg_precision": average_precision_score(y, score),
    }


baseline_te = encode_rule(te.copy())
baseline_score_te = baseline_te["queue_score"].to_numpy()

def make_models():
    return {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "decision_tree_d3": DecisionTreeClassifier(
            class_weight="balanced", max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
        ),
    }


models = make_models()

results = {"baseline_rule": evaluate(y_te, baseline_score_te)}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    results[name] = evaluate(y_te, proba)

print(f"scikit-learn {sklearn.__version__} | seed {RANDOM_STATE} everywhere")
print(f"base rate on test clients: {y_te.mean():.3f}")
for name, r in results.items():
    line = "  ".join(f"p@{k}={r[f'prec@{k}']:.3f}" for k in KS)
    print(f"{name:22s} {line}   auc={r['roc_auc']:.3f} ap={r['avg_precision']:.3f}")

scikit-learn 1.9.0 | seed 42 everywhere
base rate on test clients: 0.342
baseline_rule          p@10=0.400  p@25=0.400  p@50=0.360  p@100=0.330   auc=0.433 ap=0.307
logistic_regression    p@10=0.400  p@25=0.320  p@50=0.300  p@100=0.330   auc=0.492 ap=0.335
decision_tree_d3       p@10=0.600  p@25=0.440  p@50=0.480  p@100=0.410   auc=0.501 ap=0.345
random_forest          p@10=0.300  p@25=0.240  p@50=0.220  p@100=0.320   auc=0.515 ap=0.347


In [6]:
rows = []
rows.append({"model": "base rate (random pick)",
             **{f"prec@{k}": y_te.mean() for k in KS}, "roc_auc": np.nan, "avg_precision": np.nan})
rows.append({"model": "baseline rule (visible*slipped*depth)",
             **{f"prec@{k}": results['baseline_rule'][f'prec@{k}'] for k in KS},
             **{k: results['baseline_rule'][k] for k in ('roc_auc', 'avg_precision')}})
for name, label in [
    ("logistic_regression", "logistic regression"),
    ("decision_tree_d3", "decision tree (depth 3)"),
    ("random_forest", "random forest"),
]:
    r = results[name]
    rows.append({"model": label,
                 **{f"prec@{k}": r[f'prec@{k}'] for k in KS},
                 **{k: r[k] for k in ('roc_auc', 'avg_precision')}})

comparison = pd.DataFrame(rows)
print(comparison.round(3).to_string(index=False))

                                model  prec@10  prec@25  prec@50  prec@100  roc_auc  avg_precision
              base rate (random pick)    0.342    0.342    0.342     0.342      NaN            NaN
baseline rule (visible*slipped*depth)    0.400    0.400    0.360     0.330    0.433          0.307
                  logistic regression    0.400    0.320    0.300     0.330    0.492          0.335
              decision tree (depth 3)    0.600    0.440    0.480     0.410    0.501          0.345
                        random forest    0.300    0.240    0.220     0.320    0.515          0.347


In [7]:
best = max(("baseline_rule",) + tuple(models),
            key=lambda n: (results[n]["prec@50"], results[n]["avg_precision"], results[n]["roc_auc"]))
print("Best row by precision@50 (then average precision, then ROC-AUC):", best)

metrics_receipt = {
    "slice": "March 2026 page-level (w03 contract)",
    "rows": int(len(features)),
    "test_rows": int(len(te)),
    "test_clients": int(te["client_hash_id"].nunique()),
    "split": f"client_grouped_holdout_0.2_seed{RANDOM_STATE}",
    "base_rate_test": round(float(y_te.mean()), 3),
    "headline_metric": "precision@50 on test clients",
    "baseline_w04_full_slice_prec50": 0.56,
    "models": {name: {k: round(v, 3) for k, v in r.items()} for name, r in results.items()},
    "best_by_prec50": best,
    "leakage": "imp_trend_pct_mar / trend columns excluded; label halves never features",
    "output": "work/outputs/w05_model_metrics.json",
}
with open(OUT_DIR / "w05_model_metrics.json", "w") as f:
    json.dump(metrics_receipt, f, indent=2, sort_keys=True)
print("Metrics receipt written to work/outputs/w05_model_metrics.json")

Best row by precision@50 (then average precision, then ROC-AUC): decision_tree_d3
Metrics receipt written to work/outputs/w05_model_metrics.json


In [8]:
SEEDS = [0, 1, 2, 3, 4]
fold_rows = []
for seed in SEEDS:
    is_train, is_test, _ = client_grouped_split(features, seed=seed)
    trf = features.loc[is_train].reset_index(drop=True)
    tef = features.loc[is_test].reset_index(drop=True)
    Xtrf, Xtef = build_matrix(trf), build_matrix(tef)
    ytrf = trf["is_declining_label"].astype(int).to_numpy()
    ytef = tef["is_declining_label"].astype(int).to_numpy()
    base_score_tef = encode_rule(tef.copy())["queue_score"].to_numpy()
    row = {
        "seed": seed,
        "test_rows": len(tef),
        "base_rate": float(ytef.mean()),
        "baseline_rule_prec50": precision_at_k(ytef, base_score_tef, 50),
    }
    for name, model in make_models().items():
        model.fit(Xtrf, ytrf)
        proba = model.predict_proba(Xtef)[:, 1]
        row[f"prec50_{name}"] = precision_at_k(ytef, proba, 50)
        row[f"auc_{name}"] = roc_auc_score(ytef, proba)
    fold_rows.append(row)

folds = pd.DataFrame(fold_rows)
cols = [col for col in folds.columns if col != "seed"]
print("Model-vs-baseline robustness — precision@50 per client-grouped split (same metric, same split each time):")
print(folds.round(3).to_string(index=False))
print()
for col in cols:
    print(f"  mean {col:28s} = {folds[col].mean():.3f}   (min {folds[col].min():.3f} / max {folds[col].max():.3f})")

receipt_path = OUT_DIR / "w05_model_metrics.json"
rc = json.loads(receipt_path.read_text())
rc["repeated_grouped_mean_prec50"] = {str(k): round(float(folds[k].mean()), 3) for k in cols}
rc["repeated_seeds"] = SEEDS
receipt_path.write_text(json.dumps(rc, indent=2, sort_keys=True))
print("\nRepeated-split summary appended to w05_model_metrics.json")

Model-vs-baseline robustness — precision@50 per client-grouped split (same metric, same split each time):
 seed  test_rows  base_rate  baseline_rule_prec50  prec50_logistic_regression  auc_logistic_regression  prec50_decision_tree_d3  auc_decision_tree_d3  prec50_random_forest  auc_random_forest
    0      11594      0.098                  0.34                        0.30                    0.538                     0.46                 0.619                  0.48              0.538
    1       5871      0.304                  0.24                        0.38                    0.464                     0.34                 0.510                  0.12              0.522
    2       2584      0.198                  0.16                        0.18                    0.539                     0.26                 0.528                  0.22              0.606
    3       2332      0.195                  0.18                        0.34                    0.558                     0.36   

## 4. Errors and interpretation

A metric without error analysis is decoration. Three moves, all on the test split: (1) what the random forest leans on — tree importances, then permutation importance to catch anything suspiciously perfect; (2) where it is most wrong — accuracy by position tier and impression tier; (3) three concrete wrong cases with reasons.

In [9]:
rf = models["random_forest"]

imp = pd.DataFrame({"feature": X_tr.columns, "importance": rf.feature_importances_})
imp = imp.sort_values("importance", ascending=False)
print("Random-forest feature importance (from the fit):")
print(imp.round(4).to_string(index=False))

from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, scoring="roc_auc", random_state=RANDOM_STATE)
perm_df = pd.DataFrame({
    "feature": X_te.columns,
    "test_auc_drop_when_shuffled": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("test_auc_drop_when_shuffled", ascending=False)
print("\nPermutation importance on test (mean test-ROC-AUC drop when the feature is shuffled):")
print(perm_df.round(4).to_string(index=False))

# Logistic regression coefficients (scaled inputs, so comparable) — the readable sanity check.
lr = models["logistic_regression"].named_steps["clf"]
lr_coef = pd.Series(lr.coef_[0], index=X_tr.columns).sort_values(ascending=False)
print("\nLogistic regression standardized coefficients (positive = more decline risk):")
print(lr_coef.round(3).to_string())


Random-forest feature importance (from the fit):
                feature  importance
       avg_position_mar      0.4089
       content_age_days      0.4009
log_gsc_impressions_mar      0.1074
    engagement_rate_mar      0.0651
 days_since_last_update      0.0177
         has_engagement      0.0000



Permutation importance on test (mean test-ROC-AUC drop when the feature is shuffled):
                feature  test_auc_drop_when_shuffled    std
log_gsc_impressions_mar                       0.0307 0.0034
 days_since_last_update                       0.0033 0.0009
         has_engagement                       0.0000 0.0000
       avg_position_mar                      -0.0034 0.0057
       content_age_days                      -0.0053 0.0063
    engagement_rate_mar                      -0.0099 0.0036

Logistic regression standardized coefficients (positive = more decline risk):
avg_position_mar           0.638
days_since_last_update     0.069
engagement_rate_mar        0.052
log_gsc_impressions_mar    0.004
has_engagement            -0.001
content_age_days          -0.063


In [10]:
te2 = te.copy()
te2["rf_proba"] = rf.predict_proba(X_te)[:, 1]
te2["pred_declining"] = (te2["rf_proba"] >= 0.5).astype(int)
te2["correct"] = (te2["pred_declining"] == te2["is_declining_label"]).astype(int)

pos_bins = [-0.1, 3, 10, 20, 50, np.inf]
pos_labels = ["<=3", "4-10", "11-20", "21-50", "51+"]
te2["pos_tier"] = pd.cut(te2["avg_position_mar"], bins=pos_bins, labels=pos_labels)
print("I. Where the RF is right/wrong — by position tier (test clients):")
print(te2.groupby("pos_tier", observed=True)
        .agg(n=("content_hash_id", "size"), accuracy=("correct", "mean"),
             decline_rate=("is_declining_label", "mean"))
        .round(3).to_string())

imp_bins = [100, 300, 1000, 3000, 10000, np.inf]
imp_labels = ["100-299", "300-999", "1000-2999", "3000-9999", "10000+"]
te2["imp_tier"] = pd.cut(te2["gsc_impressions_mar"], bins=imp_bins, labels=imp_labels)
print("\nII. Where the RF is right/wrong — by impression tier (test clients):")
print(te2.groupby("imp_tier", observed=True)
        .agg(n=("content_hash_id", "size"), accuracy=("correct", "mean"),
             decline_rate=("is_declining_label", "mean"))
        .round(3).to_string())

I. Where the RF is right/wrong — by position tier (test clients):
             n  accuracy  decline_rate
pos_tier                              
<=3        562     0.673         0.297
4-10      2658     0.609         0.358
11-20      730     0.486         0.337
21-50      516     0.386         0.312
51+         29     0.517         0.379

II. Where the RF is right/wrong — by impression tier (test clients):
              n  accuracy  decline_rate
imp_tier                               
100-299    1594     0.487         0.393
300-999    1334     0.548         0.366
1000-2999   817     0.611         0.323
3000-9999   519     0.730         0.220
10000+      217     0.788         0.189


In [11]:
why0 = te2.sort_values("rf_proba", ascending=False).reset_index(drop=True)
fp = why0[why0["is_declining_label"] == 0].head(3)
fn = why0[why0["is_declining_label"] == 1].tail(3)
cols = ["gsc_impressions_mar", "avg_position_mar", "content_age_days", "days_since_last_update",
        "engagement_rate_mar", "rf_proba", "is_declining_label"]
print("False positives — RF says decline, page is flat (label=0):")
print(fp[cols].round(3).to_string(index=False))
print("\nFalse negatives — RF says safe, page actually declined (label=1):")
print(fn[cols].round(3).to_string(index=False))


def profile_topk(frame, score_col, k=50):
    top = frame.sort_values(score_col, ascending=False).head(k)
    fp_top = top[top["is_declining_label"] == 0]
    return {
        "prec@k": round(float(top["is_declining_label"].mean()), 3),
        "false_pos_share": round(float(len(fp_top) / len(top)), 3),
        "fp_mean_impressions": round(float(fp_top["gsc_impressions_mar"].mean()) if len(fp_top) else 0.0, 1),
        "fp_mean_position": round(float(fp_top["avg_position_mar"].mean()) if len(fp_top) else 0.0, 1),
    }


te2["queue_score"] = baseline_te["queue_score"].to_numpy()
print("\nTop-50 profile on the same test clients — baseline vs RF:")
print("baseline:", profile_topk(te2, "queue_score"))
print("RF:      ", profile_topk(te2, "rf_proba"))

# Show that the rule's w04 full-slice behaviour was population-local (train vs test clients).
baseline_tr = encode_rule(tr.copy())
print("\nSame rule, on its own train clients vs the held-out test clients:")
for label_, frame_, sc in [("train clients", tr, baseline_tr), ("test clients", te, baseline_te)]:
    f = frame_.copy()
    f["bs"] = sc["queue_score"].to_numpy()
    flagged = float((sc["score"] > 0).mean())
    p50 = float(f.sort_values("bs", ascending=False).head(50)["is_declining_label"].mean())
    print(f"{label_:>13}: flagged_share={flagged:.2f} | rule precision@50 = {p50:.2f}")


False positives — RF says decline, page is flat (label=0):
 gsc_impressions_mar  avg_position_mar  content_age_days  days_since_last_update  engagement_rate_mar  rf_proba  is_declining_label
              9126.0            26.426               230                       0                3.704     0.870                   0
              3444.0            16.956               158                       0                7.143     0.792                   0
              2981.0            29.433               160                       0               12.500     0.789                   0

False negatives — RF says safe, page actually declined (label=1):
 gsc_impressions_mar  avg_position_mar  content_age_days  days_since_last_update  engagement_rate_mar  rf_proba  is_declining_label
              1780.0             3.580               256                       0                0.000     0.046                   1
              2628.0             3.842               320                       0  

### Reading the errors

Read the printed tables in this order — the single seed-42 holdout first, then the repeated grouped-split table beneath it.

- **The headline: the depth-3 decision tree is the only model that wins at the top of the queue, and it wins with almost no complexity.** On the seed-42 holdout its precision@50 is 0.48 vs the rule's 0.36 (+0.12). Averaged over the five client-grouped splits the tree's mean precision@50 (0.304) beats the rule (0.232) and logistic regression (0.300), while the random forest ties the rule (0.232) despite being the heaviest model here. Where the forest pays is **ranking**: its mean test ROC-AUC (0.578) is the best of the group (decision tree 0.555, logistic regression 0.510). Practical read: the tree is the keeper for the queue; the forest is a ranking refinement. Complexity does not earn precision at the top of the queue by itself.
- **The rule weakens on unseen clients — modestly, but really.** Its full-slice precision@50 (≈0.56, w04) holds on the *train* clients (0.58) and drops to 0.36 on the held-out clients and to a **0.232 mean across the five unseen folds**, against a base rate that itself swings 0.10-0.30 per fold. It stays above random, but it is population-fitted: it concentrated on the deep-and-declining pages that crowd this March slice, and the held-out clients have fewer of those (flagged share 0.29 on train → 0.11 on test).
- **What the forest leans on (no leak) — with a humbling wrinkle.** In the fit, position (0.41) and content age (0.40) dominate and demand (0.11) is third — the same two signals the w04 audit validated. But the test permutation check flips it: on this held-out fold only **demand** clearly pays (shuffling log-impressions costs ≈ 0.03 test ROC-AUC); shuffling position, age, or engagement actually *raises* test AUC by a whisker. The forest over-learns position/age from train, and those do not transfer to every client's ranking. Nothing is suspiciously perfect, so no leak — the honest wrinkle is that a model's most-used features are not always its most transferable ones. One line of context: logistic regression assigns content age a *negative* standardized coefficient (−0.06) while the forest treats it as top importance — the two disagree, which is exactly why the fold-to-fold ranges below are wide.
- **Where the model is wrong** (test clients). Accuracy falls to ≈ 0.49 for striking-distance pages (positions 11-20) and ≈ 0.39 deep (21-50); deep demand-heavy pages are its dominant error group. The forest's highest-confidence false positives are **big, deep, flat pages** (positions ≈ 17-29, 3-9k impressions, real engagement, p ≈ 0.79-0.87) — it re-learned the exact seam the w04 skeptic warned the rule over-orders — and its clearest false negatives are **page-one pages that declined anyway** (positions ≈ 3.6-3.8, p ≈ 0.04-0.05). Both errors are demand-side: a page that slipped for one keyword, a SERP re-rank, a season — none visible to page-level features.
- **Bottom line for the strategist.** Build the queue with the **depth-3 decision tree** (best precision@50 on unseen clients, printable, built from the same five contract signals), keep the **rule** as the explainable reason-codes layer and a no-model fallback, and treat the **forest as optional** — worthwhile only if the goal is whole-list ranking, not the top of the list. Honesty to the last fold: the tree's advantage is real but modest (+0.07 mean, and it loses the fold-to-fold duel with the rule in more than one draw), so this is a recommendation, not a paradigm shift.

## Self-check

- [x] Method choice explained and matched to the question — ranking via P(decline), evaluated at precision@K (w02 metric)
- [x] Valid split design — client-grouped holdout, seeds fixed, no row-random client leakage; no time-split possible in a one-month slice
- [x] Baseline and models in the SAME table, same test clients, same metric — baseline re-scored on test only, nothing fitted to the label
- [x] Errors read, not just scored — feature importance, permutation check, accuracy by position/impression tier, three wrong cases
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Week's paper read — `docs/flyrank-seo-research-march-2026.pdf` — its ML-appendix methodology (active-subset sampling, 80/20 row splits, descriptive importance, no intervals) noted for next week's inspection
- [x] Runs top to bottom; metrics receipt at `work/outputs/w05_model_metrics.json`